# Module 4: The Transformer Block

The transformer block is the fundamental repeating unit of all modern LLMs. Each block follows a precise sequence:

```
Input
  |----> LayerNorm --> Multi-Head Attention --|
  |                                          | (add)
  |------------------------------------------|
  |----> LayerNorm --> Feed-Forward Network --|
  |                                          | (add)
  |------------------------------------------|
Output
```

This is the **Pre-Norm** architecture (used by GPT-2, LLaMA, etc.), where LayerNorm is applied *before* each sub-layer rather than after. The residual connections ensure that gradients flow cleanly through the entire stack, while LayerNorm stabilizes activations.

In this notebook we will:
1. Inspect a single transformer block and its activation statistics
2. Visualize information flow through each sub-layer
3. Compare a compact block implementation to the full one
4. Stack blocks into a mini-transformer and analyze per-layer behavior
5. Ablate individual components to understand their contributions
6. Manually examine residual magnitudes and attention patterns

## Imports and Setup

In [ ]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

torch.manual_seed(42)

# Ensure companion modules are importable
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from transformer import (
    TransformerBlock,
    MiniTransformer,
    visualize_transformer_block_flow,
    visualize_mini_transformer_analysis,
    ablate_transformer_components,
)

print(f"PyTorch version: {torch.__version__}")
print("Imports complete.")

---
## 1. Single Transformer Block

A single `TransformerBlock` contains:
- **LayerNorm + Multi-Head Attention** with a residual connection
- **LayerNorm + Feed-Forward Network** with a residual connection

Let's create one and inspect its structure.

In [ ]:
torch.manual_seed(42)

# Create a single transformer block
embed_dim = 64
num_heads = 4
ffn_dim = 256

block = TransformerBlock(embed_dim, num_heads, ffn_dim)
print(block)
print(f"\nTotal parameters: {sum(p.numel() for p in block.parameters()):,}")

In [ ]:
# Run the block on random input
torch.manual_seed(42)

batch_size = 2
seq_len = 8
x = torch.randn(batch_size, seq_len, embed_dim)

block.eval()
with torch.no_grad():
    output = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nInput norm (mean):  {x.norm(dim=-1).mean():.4f}")
print(f"Output norm (mean): {output.norm(dim=-1).mean():.4f}")
print(f"\nAttention weights shape: {block.attn_weights.shape}")
print("  -> (batch, seq_len, seq_len) -- each position attends to all others")

The input and output have **identical shapes** -- this is what makes transformer blocks stackable. The residual connections keep the output norm close to the input norm.

## 2. Activation Statistics: Attention vs FFN

Each block has two major sub-layers. Let's measure how much each one contributes to the output.

In [ ]:
torch.manual_seed(42)

x = torch.randn(batch_size, seq_len, embed_dim)

block.eval()
with torch.no_grad():
    stats = block.get_activation_stats(x)

print("Activation Statistics")
print("=" * 40)
print(f"Input norm:           {stats['input_norm']:.4f}")
print(f"Input std:            {stats['input_std']:.4f}")
print(f"Attention contrib:    {stats['attn_contribution']:.4f}")
print(f"FFN contribution:     {stats['ffn_contribution']:.4f}")
print(f"\nAttn / FFN ratio:     {stats['attn_contribution'] / (stats['ffn_contribution'] + 1e-8):.4f}")
print(f"\nInterpretation:")
if stats['attn_contribution'] > stats['ffn_contribution']:
    print("  Attention contributes more signal than FFN at this layer.")
else:
    print("  FFN contributes more signal than attention at this layer.")

In practice, the balance between attention and FFN shifts across layers. Early layers tend to have stronger attention contributions (gathering context), while later layers often show stronger FFN contributions (transforming representations).

---
## 3. Information Flow Visualization

The following visualization traces a tensor through every stage of the transformer block:
1. **Input** -- the raw representation
2. **Attention output** -- what the attention sub-layer adds
3. **After attention + residual** -- input blended with attention
4. **FFN output** -- what the feed-forward sub-layer adds
5. **Final output** -- the complete block output
6. **Activation norms** -- magnitude at each stage

In [ ]:
torch.manual_seed(42)
visualize_transformer_block_flow()

Notice how the attention and FFN outputs have relatively **small norms** compared to the input. The residual connection means the block output is dominated by the input, with the sub-layers making incremental adjustments. This is critical for training stability -- each block makes a small, learnable correction rather than completely rewriting the representation.

---
## 4. Compact Block Implementation

The `Block` class in `block.py` is a minimal implementation that captures the same architecture in fewer lines. It uses:
- A custom `MHA` (Multi-Head Attention) from Module 3
- GELU activation (used by GPT-2, BERT) instead of ReLU
- 4x expansion in the FFN (standard practice)
- **Post-Norm** ordering (norm after residual, as in the original Transformer paper)

Let's inspect the source and compare.

In [ ]:
# Display the compact Block source code
import inspect

block_source = open("block.py").read()
print(block_source)

In [ ]:
# Compare architectures side-by-side
print("TransformerBlock (transformer.py)          | Block (block.py)")
print("=" * 66)
print("Pre-Norm (LN before sub-layers)            | Post-Norm (LN after residual)")
print("nn.MultiheadAttention (PyTorch built-in)    | Custom MHA from Module 3")
print("ReLU activation in FFN                     | GELU activation in FFN")
print("Configurable FFN dim                       | Fixed 4x expansion")
print("Dropout layers included                    | No dropout")
print("Stores attention weights for analysis       | Returns weights from MHA")

Both implementations encode the same core idea: **residual(attention) + residual(FFN)**. The differences are in normalization ordering, activation functions, and configurability. Modern LLMs typically use Pre-Norm with SwiGLU or GELU.

---
## 5. Mini-Transformer: Stacking Blocks

A full language model is just:
1. Token + positional embeddings
2. A stack of N transformer blocks
3. Final LayerNorm + linear projection to vocabulary

Let's build a 4-block mini-transformer and run it.

In [ ]:
torch.manual_seed(42)

model = MiniTransformer(
    vocab_size=256,
    embed_dim=64,
    num_heads=4,
    num_blocks=4,
    ffn_dim=256,
)
model.eval()

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 6. Forward Pass: Token IDs to Logits

In [ ]:
torch.manual_seed(42)

# Random token IDs (simulating a byte-level tokenizer with vocab_size=256)
input_ids = torch.randint(0, 256, (2, 16))  # batch=2, seq_len=16
print(f"Input token IDs shape: {input_ids.shape}")
print(f"Sample tokens: {input_ids[0, :8].tolist()}")

with torch.no_grad():
    logits = model(input_ids)

print(f"\nLogits shape: {logits.shape}")
print(f"  -> (batch_size={logits.shape[0]}, seq_len={logits.shape[1]}, vocab_size={logits.shape[2]})")

# Convert last position to probabilities
probs = F.softmax(logits[0, -1, :], dim=-1)
print(f"\nProbability distribution over vocab (last token, batch 0):")
print(f"  Sum of probabilities: {probs.sum():.4f}")
print(f"  Max probability:      {probs.max():.4f} (token {probs.argmax().item()})")
print(f"  Min probability:      {probs.min():.6f}")
print(f"  Entropy:              {-(probs * probs.log()).sum():.4f} (max={np.log(256):.4f})")

With random (untrained) weights, the probabilities are nearly uniform across the vocabulary. After training, the model would concentrate probability mass on likely next tokens.

---
## 7. Per-Layer Analysis

One of the most informative diagnostics is tracking how activations evolve layer by layer. We measure:
- **Input norm**: How large the representations are entering each block
- **Attention contribution**: How much the attention sub-layer adds
- **FFN contribution**: How much the FFN sub-layer adds

In [ ]:
torch.manual_seed(42)

input_ids = torch.randint(0, 256, (2, 16))

with torch.no_grad():
    layer_stats = model.analyze_layer_outputs(input_ids)

print(f"{'Layer':<8} {'Input Norm':<14} {'Input Std':<14} {'Attn Contrib':<14} {'FFN Contrib':<14} {'Attn/FFN':>10}")
print("=" * 78)
for s in layer_stats:
    ratio = s['attn_contribution'] / (s['ffn_contribution'] + 1e-8)
    print(f"{s['layer']:<8} {s['input_norm']:<14.4f} {s['input_std']:<14.4f} "
          f"{s['attn_contribution']:<14.4f} {s['ffn_contribution']:<14.4f} {ratio:>10.4f}")

## 8. Layer Analysis Visualization

In [ ]:
torch.manual_seed(42)
visualize_mini_transformer_analysis()

The three plots reveal the internal dynamics of the model:
- **Input norm by layer**: Shows whether representations grow, shrink, or stay stable. LayerNorm helps keep this controlled.
- **Component contributions**: The relative magnitude of what attention and FFN add at each layer.
- **Attn/FFN ratio**: Whether a layer is "attention-dominated" (ratio > 1) or "FFN-dominated" (ratio < 1).

---
## 9. Component Ablation Study

What happens when we remove key components from the transformer block? This ablation study tests five configurations:
1. **Full Block** -- all components present (baseline)
2. **No Residuals** -- remove skip connections
3. **No LayerNorm** -- remove normalization
4. **Attn Only** -- skip the FFN sub-layer
5. **FFN Only** -- skip the attention sub-layer

In [ ]:
torch.manual_seed(42)
ablate_transformer_components()

Key observations from the ablation:
- **Removing residuals** causes signal degradation -- the output norm drops because information is lost at each stage.
- **Removing LayerNorm** may cause the norm to explode or collapse depending on initialization, making training unstable.
- **Attention-only** and **FFN-only** blocks still function but lose expressiveness. Attention handles inter-token mixing, while FFN handles per-token transformation.

---
## 10. Manual Analysis: Residual Connection Magnitude Over Layers

Residual connections are the backbone of deep transformers. Let's manually trace the magnitude of what each sub-layer adds relative to the residual stream.

In [ ]:
torch.manual_seed(42)

model.eval()
input_ids = torch.randint(0, 256, (2, 16))

# Manually trace through the model
with torch.no_grad():
    # Embed
    x = model.token_embed(input_ids)
    positions = torch.arange(input_ids.shape[1]).unsqueeze(0)
    x = x + model.pos_embed(positions)

    residual_norms = []
    attn_delta_norms = []
    ffn_delta_norms = []

    for i, blk in enumerate(model.blocks):
        residual_norms.append(x.norm(dim=-1).mean().item())

        # Attention sub-layer
        x_norm = blk.norm1(x)
        attn_out, _ = blk.attention(x_norm, x_norm, x_norm, need_weights=False)
        attn_delta_norms.append(attn_out.norm(dim=-1).mean().item())
        x = x + attn_out

        # FFN sub-layer
        x_norm2 = blk.norm2(x)
        ffn_out = blk.ffn(x_norm2)
        ffn_delta_norms.append(ffn_out.norm(dim=-1).mean().item())
        x = x + ffn_out

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
layers = range(len(model.blocks))

axes[0].plot(layers, residual_norms, 'o-', linewidth=2, label='Residual stream')
axes[0].plot(layers, attn_delta_norms, 's--', linewidth=2, label='Attention delta')
axes[0].plot(layers, ffn_delta_norms, '^--', linewidth=2, label='FFN delta')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Mean Norm')
axes[0].set_title('Residual Stream vs Sub-Layer Contributions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ratio: how much each sub-layer changes the residual
attn_ratios = [a / r for a, r in zip(attn_delta_norms, residual_norms)]
ffn_ratios = [f / r for f, r in zip(ffn_delta_norms, residual_norms)]

axes[1].bar(np.array(list(layers)) - 0.15, attn_ratios, width=0.3, label='Attn / Residual', alpha=0.8)
axes[1].bar(np.array(list(layers)) + 0.15, ffn_ratios, width=0.3, label='FFN / Residual', alpha=0.8)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Delta / Residual Ratio')
axes[1].set_title('Relative Sub-Layer Impact')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nAs depth increases, the residual stream norm grows while the relative")
print("impact of each sub-layer (delta/residual ratio) typically decreases.")
print("This means deeper layers make finer adjustments to a larger signal.")

---
## 11. Manual Analysis: Attention Weight Patterns Per Block

Each transformer block learns different attention patterns. Let's visualize the attention weights from every block in our mini-transformer.

In [ ]:
torch.manual_seed(42)

model.eval()
input_ids = torch.randint(0, 256, (1, 12))  # single example, 12 tokens

# Forward pass to populate attention weights
with torch.no_grad():
    # Manually pass through to capture attention weights at each block
    x = model.token_embed(input_ids)
    positions = torch.arange(input_ids.shape[1]).unsqueeze(0)
    x = x + model.pos_embed(positions)

    all_attn_weights = []
    for blk in model.blocks:
        x = blk(x, need_weights=True)
        all_attn_weights.append(blk.attn_weights[0].numpy())  # (seq, seq)

# Plot attention patterns for each block
num_blocks = len(all_attn_weights)
fig, axes = plt.subplots(1, num_blocks, figsize=(4 * num_blocks, 4))

for i, (ax, weights) in enumerate(zip(axes, all_attn_weights)):
    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=weights.max())
    ax.set_title(f'Block {i}')
    ax.set_xlabel('Key position')
    if i == 0:
        ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Attention Weight Patterns by Block', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Analyze patterns
print("\nAttention pattern analysis:")
print("=" * 50)
for i, weights in enumerate(all_attn_weights):
    diag_mean = np.diag(weights).mean()
    entropy = -(weights * np.log(weights + 1e-10)).sum(axis=-1).mean()
    max_attn = weights.max()
    print(f"Block {i}: diag_mean={diag_mean:.4f}  entropy={entropy:.4f}  max_weight={max_attn:.4f}")

Even with random weights, we can observe structural patterns:
- **Diagonal attention** (attending to self) is common in early layers
- **Diffuse attention** (high entropy) means the layer is mixing information broadly
- **Concentrated attention** (low entropy, high max weight) means the layer is focused on specific positions

In trained models, these patterns become highly specialized: some heads attend to previous tokens, others to syntactically related words, and some implement positional patterns.

---
## 12. Parameter Count Comparison

Let's break down where the parameters live in a transformer block and compare the two implementations.

In [ ]:
torch.manual_seed(42)

# Single block parameter breakdown
block = TransformerBlock(64, 4, 256)

param_groups = {
    'LayerNorm (x2)': sum(p.numel() for n, p in block.named_parameters() if 'norm' in n),
    'Attention (QKV + Out)': sum(p.numel() for n, p in block.named_parameters() if 'attention' in n),
    'FFN (2 linear layers)': sum(p.numel() for n, p in block.named_parameters() if 'ffn' in n),
}
total_block = sum(param_groups.values())

print("TransformerBlock Parameter Breakdown (embed=64, heads=4, ffn=256)")
print("=" * 60)
for name, count in param_groups.items():
    print(f"  {name:<30}: {count:>8,}  ({100 * count / total_block:5.1f}%)")
print(f"  {'TOTAL':<30}: {total_block:>8,}")

print()

# Full model parameter breakdown
model = MiniTransformer(vocab_size=256, embed_dim=64, num_heads=4, num_blocks=4, ffn_dim=256)
model_groups = {
    'Token embeddings': sum(p.numel() for n, p in model.named_parameters() if 'token_embed' in n),
    'Position embeddings': sum(p.numel() for n, p in model.named_parameters() if 'pos_embed' in n),
    'Transformer blocks (x4)': sum(p.numel() for n, p in model.named_parameters() if 'blocks' in n),
    'Final norm + head': sum(p.numel() for n, p in model.named_parameters()
                             if 'blocks' not in n and 'embed' not in n),
}
total_model = sum(model_groups.values())

print("MiniTransformer Parameter Breakdown (vocab=256, blocks=4)")
print("=" * 60)
for name, count in model_groups.items():
    print(f"  {name:<30}: {count:>8,}  ({100 * count / total_model:5.1f}%)")
print(f"  {'TOTAL':<30}: {total_model:>8,}")

In [ ]:
# Visualize parameter distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Block breakdown
colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].pie(param_groups.values(), labels=param_groups.keys(), autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title(f'Single Block ({total_block:,} params)')

# Model breakdown
colors_model = ['#9C27B0', '#E91E63', '#3F51B5', '#009688']
axes[1].pie(model_groups.values(), labels=model_groups.keys(), autopct='%1.1f%%',
            colors=colors_model, startangle=90)
axes[1].set_title(f'MiniTransformer ({total_model:,} params)')

plt.suptitle('Where Do the Parameters Live?', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nThe FFN accounts for the majority of block parameters.")
print(f"In the full model, the 4 transformer blocks contain "
      f"{100 * model_groups['Transformer blocks (x4)'] / total_model:.1f}% of all parameters.")

This parameter distribution holds at scale: in GPT-3 (175B parameters), the FFN layers account for roughly 2/3 of all parameters, while attention accounts for about 1/3. The embedding and output layers are a small fraction.

---
## Key Insights

**1. The transformer block is a residual correction machine.** Each block adds a small update to the residual stream rather than rewriting it. This design enables gradient flow through hundreds of layers.

**2. Pre-Norm vs Post-Norm matters for training stability.** Pre-Norm (LN before sub-layers) is preferred in modern architectures because it keeps gradients well-behaved without careful learning rate warmup.

**3. Attention and FFN serve complementary roles.** Attention mixes information across sequence positions (inter-token), while FFN transforms each position independently (intra-token). Both are necessary -- ablating either one degrades the model.

**4. Residual connections are non-negotiable.** Without skip connections, signal degrades rapidly through deep networks. The residual stream acts as a "memory highway" that preserves information.

**5. LayerNorm prevents activation explosion.** Without normalization, the activation norms can grow or collapse unpredictably, making optimization extremely difficult.

**6. The FFN dominates parameter count.** With its 4x expansion factor, the FFN typically contains ~2/3 of a block's parameters. This is why techniques like Mixture of Experts (Module 8) target the FFN layer specifically.

**7. Deeper layers make finer adjustments.** As the residual stream grows in norm through the network, the relative contribution of each sub-layer decreases. Early layers set the broad context; later layers refine it.